STAT 453: Deep Learning (Spring 2021)  
Instructor: Sebastian Raschka (sraschka@wisc.edu)  

Course website: http://pages.stat.wisc.edu/~sraschka/teaching/stat453-ss2021/  
GitHub repository: https://github.com/rasbt/stat453-deep-learning-ss21

---

# Fine-tuning BERT for Movie Review Classfification

Based on https://huggingface.co/transformers/custom_datasets.html

> DistilBERT is a small, fast, cheap and light Transformer model trained by distilling BERT base. It has 40% less parameters than bert-base-uncased , runs 60% faster while preserving over 95% of BERT's performances as measured on the GLUE language understanding benchmark.

You can install the transformer library via

    pip install transformers

In [2]:
%load_ext watermark
%watermark -a 'Dr. T' -v -p torch,transformers

Author: Dr. T

Python implementation: CPython
Python version       : 3.10.20
IPython version      : 8.39.0

torch       : 2.6.0+cu124
transformers: 5.15.0



In [3]:
import gzip
import shutil
import time

import pandas as pd
import requests
import torch
import torch.nn.functional as F

import transformers
from transformers import DistilBertTokenizerFast
from transformers import DistilBertForSequenceClassification

/home/atwumasi/miniconda3/envs/mscs635/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## General Settings

In [4]:
torch.backends.cudnn.deterministic = True
RANDOM_SEED = 123
torch.manual_seed(RANDOM_SEED)
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

NUM_EPOCHS = 3

## Download Dataset

The following cells will download the IMDB movie review dataset (http://ai.stanford.edu/~amaas/data/sentiment/) for positive-negative sentiment classification in as CSV-formatted file:

In [5]:
url = "https://github.com/rasbt/python-machine-learning-book-3rd-edition/raw/master/ch08/movie_data.csv.gz"
filename = url.split("/")[-1]

with open(filename, "wb") as f:
    r = requests.get(url)
    f.write(r.content)

with gzip.open('movie_data.csv.gz', 'rb') as f_in:
    with open('movie_data.csv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

Check that the dataset looks okay:

In [6]:
df = pd.read_csv('movie_data.csv')
df.head()

,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0
3,hi for all the people who have seen this wonde...,1
4,"I recently bought the DVD, forgetting just how...",0


In [7]:
df.shape

(50000, 2)

## Split Dataset into Train/Validation/Test

In [8]:
train_texts = df.iloc[:35000]['review'].values
train_labels = df.iloc[:35000]['sentiment'].values

valid_texts = df.iloc[35000:40000]['review'].values
valid_labels = df.iloc[35000:40000]['sentiment'].values

test_texts = df.iloc[40000:]['review'].values
test_labels = df.iloc[40000:]['sentiment'].values

## Tokenization

In [9]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

In [10]:
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True)
valid_encodings = tokenizer(list(valid_texts), truncation=True, padding=True)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True)

In [11]:
train_encodings[0]

Encoding(num_tokens=512, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

## Dataset Class and Loaders

In [12]:
class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = IMDbDataset(train_encodings, train_labels)
valid_dataset = IMDbDataset(valid_encodings, valid_labels)
test_dataset = IMDbDataset(test_encodings, test_labels)

In [13]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=16, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=16, shuffle=False)

## Load Model

In [14]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased')
model.to(DEVICE)
model.train()

optim = torch.optim.Adam(model.parameters(), lr=5e-5)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 317.43it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Train Model

In [15]:
def compute_accuracy(model, data_loader, device):

    with torch.no_grad():

        correct_pred, num_examples = 0, 0

        for batch_idx, batch in enumerate(data_loader):

            ### Prepare data
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss, logits = outputs['loss'], outputs['logits']

            _, predicted_labels = torch.max(logits, 1)

            num_examples += labels.size(0)

            correct_pred += (predicted_labels == labels).sum()
    return correct_pred.float()/num_examples * 100

In [16]:
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    
    model.train()
    
    for batch_idx, batch in enumerate(train_loader):
        
        ### Prepare data
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        ### Forward
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss, logits = outputs['loss'], outputs['logits']
        
        ### Backward
        optim.zero_grad()
        loss.backward()
        optim.step()
        
        ### Logging
        if not batch_idx % 250:
            print (f'Epoch: {epoch+1:04d}/{NUM_EPOCHS:04d} | '
                   f'Batch {batch_idx:04d}/{len(train_loader):04d} | '
                   f'Loss: {loss:.4f}')
            
    model.eval()

    with torch.set_grad_enabled(False):
        print(f'training accuracy: '
              f'{compute_accuracy(model, train_loader, DEVICE):.2f}%'
              f'\nvalid accuracy: '
              f'{compute_accuracy(model, valid_loader, DEVICE):.2f}%')
        
    print(f'Time elapsed: {(time.time() - start_time)/60:.2f} min')
    
print(f'Total Training Time: {(time.time() - start_time)/60:.2f} min')
print(f'Test accuracy: {compute_accuracy(model, test_loader, DEVICE):.2f}%')

Epoch: 0001/0003 | Batch 0000/2188 | Loss: 0.6806
Epoch: 0001/0003 | Batch 0250/2188 | Loss: 0.1178
Epoch: 0001/0003 | Batch 0500/2188 | Loss: 0.3492
Epoch: 0001/0003 | Batch 0750/2188 | Loss: 0.0337
Epoch: 0001/0003 | Batch 1000/2188 | Loss: 0.1073
Epoch: 0001/0003 | Batch 1250/2188 | Loss: 0.0191
Epoch: 0001/0003 | Batch 1500/2188 | Loss: 0.1893
Epoch: 0001/0003 | Batch 1750/2188 | Loss: 0.1335
Epoch: 0001/0003 | Batch 2000/2188 | Loss: 0.1616
training accuracy: 96.09%
valid accuracy: 92.10%
Time elapsed: 8.49 min
Epoch: 0002/0003 | Batch 0000/2188 | Loss: 0.0803
Epoch: 0002/0003 | Batch 0250/2188 | Loss: 0.0473
Epoch: 0002/0003 | Batch 0500/2188 | Loss: 0.0582
Epoch: 0002/0003 | Batch 0750/2188 | Loss: 0.0054
Epoch: 0002/0003 | Batch 1000/2188 | Loss: 0.1791
Epoch: 0002/0003 | Batch 1250/2188 | Loss: 0.0830
Epoch: 0002/0003 | Batch 1500/2188 | Loss: 0.0759
Epoch: 0002/0003 | Batch 1750/2188 | Loss: 0.1925
Epoch: 0002/0003 | Batch 2000/2188 | Loss: 0.5984
training accuracy: 98.83%
va

## Save Model

Persist the fine-tuned weights to disk so future kernel sessions can skip retraining (see the "Load Saved Model" cell below).

In [17]:
SAVED_MODEL_DIR = './saved_model'

model.save_pretrained(SAVED_MODEL_DIR)
tokenizer.save_pretrained(SAVED_MODEL_DIR)
print(f'Saved fine-tuned model and tokenizer to {SAVED_MODEL_DIR}')

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.01it/s]

Saved fine-tuned model and tokenizer to ./saved_model


## Run Inference on Your Own Text

If you just ran training above, `model` is already fine-tuned and in memory — skip straight to the `predict_sentiment` cell below.

If you're starting a **fresh kernel** instead: run the imports cell and the General Settings cell near the top of this notebook (fast, no download/training needed), then run the "Load Saved Model" cell below to load your previously saved weights instead of retraining from scratch.

In [18]:
### Load Saved Model (skip this cell if `model` is already trained/in memory)

SAVED_MODEL_DIR = './saved_model'

if 'model' not in globals():
    import os
    if os.path.isdir(SAVED_MODEL_DIR):
        print(f'No model in memory — loading fine-tuned weights from {SAVED_MODEL_DIR} ...')
        tokenizer = DistilBertTokenizerFast.from_pretrained(SAVED_MODEL_DIR)
        model = DistilBertForSequenceClassification.from_pretrained(SAVED_MODEL_DIR)
        model.to(DEVICE)
    else:
        raise RuntimeError(
            f"No model in memory and no saved model found at '{SAVED_MODEL_DIR}'. "
            "Run the training cells above first (that also saves the model for next time)."
        )
else:
    print('Using the model already in memory.')

model.eval()

Using the model already in memory.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [19]:
def predict_sentiment(text, model=model, tokenizer=tokenizer, device=DEVICE):
    model.eval()

    encoding = tokenizer(text, truncation=True, padding=True, return_tensors='pt')
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        probas = F.softmax(outputs['logits'], dim=1)

    predicted_label = torch.argmax(probas, dim=1).item()
    confidence = probas[0, predicted_label].item()

    label_name = 'positive' if predicted_label == 1 else 'negative'
    return label_name, confidence

In [20]:
my_reviews = [
    "This movie was absolutely fantastic, I loved every minute of it.",
    "Waste of two hours. The plot made no sense and the acting was wooden.",
    "It was okay, not great but not terrible either.",
]

for review in my_reviews:
    label, confidence = predict_sentiment(review)
    print(f'{label:>8s} ({confidence*100:.1f}%): {review}')

positive (99.7%): This movie was absolutely fantastic, I loved every minute of it.
negative (100.0%): Waste of two hours. The plot made no sense and the acting was wooden.
negative (99.6%): It was okay, not great but not terrible either.
